In [32]:
from pathlib import Path

BASE_DIR = Path("/Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs")

print("Directories in runs:")
for p in BASE_DIR.iterdir():
    if p.is_dir():
        print(" ", p)

Directories in runs:
  /Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_FAST_1p5Msun_Z0p0142


In [9]:
# ============================================================
# USER-EDITABLE CO WHITE DWARF SETTINGS
# ============================================================

# ---------- STAR ----------
MASS_MSUN = 1.5
INITIAL_Z = 0.0142

# ---------- SPEED / TERMINATION ----------
MAX_MODEL_NUMBER = 30000

# Stop removing the envelope when log10(Teff) reaches this value
LOG_TEFF_LIMIT = 3.8

# ---------- MESA NUMERICAL SETTINGS ----------
MESH_DELTA_COEFF = 1.0
VARCONTROL_TARGET = 0.01
MIN_TIMESTEP_LIMIT = 1e-8
MAX_YEARS_FOR_TIMESTEP = 1e6

# ---------- OUTPUT ----------
HISTORY_INTERVAL = 100
PROFILE_INTERVAL = 100
TERMINAL_INTERVAL = 100

# ---------- RUN CONTROL ----------
OVERWRITE_RUN = True
RUN_MESA = True


print("======================================")
print("CO-WD FAST RUN SETTINGS")
print("======================================")
print(f"Initial mass:        {MASS_MSUN} Msun")
print(f"Initial metallicity: Z = {INITIAL_Z}")
print(f"Max model number:    {MAX_MODEL_NUMBER}")
print(f"log(Teff) limit:     {LOG_TEFF_LIMIT}")
print(f"Mesh coefficient:    {MESH_DELTA_COEFF}")
print(f"Varcontrol target:   {VARCONTROL_TARGET}")
print(f"Max timestep:        {MAX_YEARS_FOR_TIMESTEP} yr")
print(f"History interval:    {HISTORY_INTERVAL}")
print(f"Profile interval:    {PROFILE_INTERVAL}")


# ============================================================
# PATHS
# ============================================================

import os
import re
import shutil
import subprocess
from pathlib import Path

# MESA installation
os.environ["MESA_DIR"] = "/Users/mchau/mesa"

# ============================================================
# NEW FAST WD INLIST SOURCE
# ============================================================

PROJECT_ROOT = Path("/Users/mchau/mesa_tutorial")

WD_DIR = (
    PROJECT_ROOT
    / "WHITE_DWARF"
    / "WD"
    / "make_co_wd_fast"
)

# ============================================================
# OUTPUT DIRECTORY
# ============================================================

RUN_NAME = f"CO_WD_FAST_{MASS_MSUN:g}Msun_Z{INITIAL_Z:g}"

RUN_DIR = PROJECT_ROOT / "WHITE_DWARF" / "WD" / "runs" / RUN_NAME

print()
print("MESA_DIR:       ", os.environ["MESA_DIR"])
print("Fast WD source: ", WD_DIR)
print("Run directory:  ", RUN_DIR)

print("Fast WD source exists:", WD_DIR.exists())

if WD_DIR.exists():
    print("\nFiles in fast WD source:")
    for f in sorted(WD_DIR.iterdir()):
        print(" ", f.name)

CO-WD FAST RUN SETTINGS
Initial mass:        1.5 Msun
Initial metallicity: Z = 0.0142
Max model number:    30000
log(Teff) limit:     3.8
Mesh coefficient:    1.0
Varcontrol target:   0.01
Max timestep:        1000000.0 yr
History interval:    100
Profile interval:    100

MESA_DIR:        /Users/mchau/mesa
Fast WD source:  /Users/mchau/mesa_tutorial/WHITE_DWARF/WD/make_co_wd_fast
Run directory:   /Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_FAST_1.5Msun_Z0.0142
Fast WD source exists: True

Files in fast WD source:
  .gitignore
  .ipynb_checkpoints
  clean
  co_core.mod
  history_columns.list
  inlist
  inlist_common
  inlist_remove_env
  inlist_remove_env_header
  inlist_settle
  inlist_settle_header
  make
  mk
  profile_columns.list
  re
  rn
  src


In [10]:
# ============================================================
# CREATE CUSTOM FAST WD RUN DIRECTORY
# ============================================================

mass_label = f"{MASS_MSUN:.1f}".replace(".", "p")
z_label = f"{INITIAL_Z:.4f}".replace(".", "p")

RUN_DIR = (
    PROJECT_ROOT
    / "WHITE_DWARF"
    / "WD"
    / "runs"
    / f"CO_WD_FAST_{mass_label}Msun_Z{z_label}"
)

print("RUN_DIR =", RUN_DIR)

# ------------------------------------------------------------
# Remove previous run if requested
# ------------------------------------------------------------

if RUN_DIR.exists():

    if OVERWRITE_RUN:
        print("Removing previous run...")
        shutil.rmtree(RUN_DIR)

    else:
        raise FileExistsError(
            f"{RUN_DIR} already exists.\n"
            "Set OVERWRITE_RUN = True or choose different parameters."
        )

RUN_DIR.mkdir(parents=True)

print("Created:", RUN_DIR)


# ============================================================
# CHECK FAST WD SOURCE
# ============================================================

if not WD_DIR.exists():
    raise FileNotFoundError(
        f"\nFast WD source directory not found:\n{WD_DIR}"
    )

print("\nFast WD source confirmed:")
print(" ", WD_DIR)


# ============================================================
# COPY MESA EXECUTABLE
# ============================================================

# Use the rebuilt MESA executable with HDF5 310.
STAR_SOURCE = Path("/Users/mchau/mesa/star/test/star")
STAR_DEST = RUN_DIR / "star"

if not STAR_SOURCE.exists():
    raise FileNotFoundError(
        f"New MESA star executable not found:\n{STAR_SOURCE}"
    )

shutil.copy2(STAR_SOURCE, STAR_DEST)
STAR_DEST.chmod(0o755)

print("\nCopied MESA executable:")
print("  From:", STAR_SOURCE)
print("  To:  ", STAR_DEST)


# ============================================================
# VERIFY HDF5 VERSION
# ============================================================

check = subprocess.run(
    ["otool", "-L", str(STAR_DEST)],
    capture_output=True,
    text=True
)

print("\n===== STAR HDF5 LIBRARIES =====")

for line in check.stdout.splitlines():
    if "hdf5" in line.lower():
        print(line)

if "libhdf5_fortran.310.dylib" not in check.stdout:
    raise RuntimeError(
        "\nWRONG MESA EXECUTABLE!\n"
        "Expected libhdf5_fortran.310.dylib, "
        "but the copied star executable is not linked to HDF5 310."
    )

print("\n✓ Correct MESA executable confirmed: HDF5 310")


# ============================================================
# COPY FAST WD INLISTS / FILES
# ============================================================

print("\n===== COPYING FAST WD SETUP =====")

for item in WD_DIR.iterdir():

    # Don't copy an old compiled executable from the source
    if item.name == "star":
        continue

    destination = RUN_DIR / item.name

    if item.is_file():
        shutil.copy2(item, destination)
        print("Copied:", item.name)

    elif item.is_dir():
        shutil.copytree(item, destination)
        print("Copied directory:", item.name)


# ============================================================
# SHOW COPIED INLISTS
# ============================================================

print("\n===== FAST WD INLISTS =====")

for f in sorted(RUN_DIR.glob("inlist*")):
    print(" ", f.name)


# ============================================================
# DONE
# ============================================================

print("\n============================================================")
print("FAST CO-WD SETUP COPIED SUCCESSFULLY")
print("============================================================")
print("Source:", WD_DIR)
print("Run:   ", RUN_DIR)

RUN_DIR = /Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_FAST_1p5Msun_Z0p0142
Removing previous run...
Created: /Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_FAST_1p5Msun_Z0p0142

Fast WD source confirmed:
  /Users/mchau/mesa_tutorial/WHITE_DWARF/WD/make_co_wd_fast

Copied MESA executable:
  From: /Users/mchau/mesa/star/test/star
  To:   /Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_FAST_1p5Msun_Z0p0142/star

===== STAR HDF5 LIBRARIES =====
	@rpath/lib/libhdf5_fortran.310.dylib (compatibility version 314.0.0, current version 314.2.0)
	@rpath/lib/libhdf5.310.dylib (compatibility version 316.0.0, current version 316.1.0)

✓ Correct MESA executable confirmed: HDF5 310

===== COPYING FAST WD SETUP =====
Copied: history_columns.list
Copied: inlist_settle_header
Copied: co_core.mod
Copied: profile_columns.list
Copied: inlist_remove_env_header
Copied: inlist_settle
Copied: rn
Copied: inlist_common
Copied: clean
Copied: .gitignore
Copied: mk
Copied directory: make
Copied: i

In [39]:
from pathlib import Path
import re

RUN_DIR = Path(
    "/Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_FAST_1p5Msun_Z0p0142"
)

# ============================================================
# APPLY EDITABLE SETTINGS TO FAST MESA INLISTS
# ============================================================

def replace_parameter(file_path, parameter, value, required=False):
    """
    Replace a MESA namelist parameter while preserving indentation.

    Returns:
        number of replacements
    """

    if not file_path.exists():
        if required:
            raise FileNotFoundError(
                f"Required inlist not found:\n{file_path}"
            )
        print(f"Skipping missing file: {file_path.name}")
        return 0

    text = file_path.read_text()

    # Match a parameter at the beginning of a line.
    pattern = (
        rf"(^\s*{re.escape(parameter)}\s*=\s*)"
        r"[-+0-9.eEdD]+"
    )

    new_text, n = re.subn(
        pattern,
        rf"\g<1>{value}",
        text,
        flags=re.MULTILINE
    )

    if n == 0:
        print(f"  WARNING: {parameter} not found in {file_path.name}")
    else:
        print(f"  {file_path.name}: {parameter} = {value}")

    file_path.write_text(new_text)

    return n


# ============================================================
# INLIST_COMMON
# ============================================================

common = RUN_DIR / "inlist_common"

print("\n======================================")
print("UPDATING inlist_common")
print("======================================")

replace_parameter(
    common,
    "initial_mass",
    f"{MASS_MSUN:.6g}d0",
    required=True
)

replace_parameter(
    common,
    "initial_z",
    f"{INITIAL_Z:.6g}d0",
    required=True
)

replace_parameter(
    common,
    "mesh_delta_coeff",
    f"{MESH_DELTA_COEFF}",
    required=False
)

replace_parameter(
    common,
    "varcontrol_target",
    f"{VARCONTROL_TARGET}",
    required=False
)

replace_parameter(
    common,
    "max_years_for_timestep",
    f"{MAX_YEARS_FOR_TIMESTEP:.6g}",
    required=False
)

replace_parameter(
    common,
    "history_interval",
    f"{HISTORY_INTERVAL}",
    required=False
)

replace_parameter(
    common,
    "terminal_interval",
    f"{TERMINAL_INTERVAL}",
    required=False
)


# ============================================================
# INLIST_REMOVE_ENV
# ============================================================

remove_env = RUN_DIR / "inlist_remove_env"

print("\n======================================")
print("UPDATING inlist_remove_env")
print("======================================")

replace_parameter(
    remove_env,
    "max_model_number",
    f"{MAX_MODEL_NUMBER}",
    required=False
)

replace_parameter(
    remove_env,
    "log_Teff_upper_limit",
    f"{LOG_TEFF_LIMIT}",
    required=False
)

replace_parameter(
    remove_env,
    "min_timestep_limit",
    f"{MIN_TIMESTEP_LIMIT:.6g}",
    required=False
)

from pathlib import Path
import re

remove_env = RUN_DIR / "inlist_remove_env"

text = remove_env.read_text()

text, n = re.subn(
    r"(^\s*max_model_number\s*=\s*)\d+",
    r"\g<1>10000",
    text,
    flags=re.MULTILINE
)

print("Replacements:", n)

remove_env.write_text(text)

# Verify
for line in text.splitlines():
    if "max_model_number" in line:
        print(line)

import re
from pathlib import Path

settle = RUN_DIR / "inlist_settle"

text = settle.read_text()

# Example: change max_model_number
text, n = re.subn(
    r"(^\s*max_model_number\s*=\s*)\d+",
    r"\g<1>10000",
    text,
    flags=re.MULTILINE
)

print("Replacements:", n)

if n == 0:
    print("WARNING: max_model_number was not found.")
else:
    settle.write_text(text)
    print("Updated:", settle)

    for line in text.splitlines():
        if "max_model_number" in line:
            print(line)


# ============================================================
# FINAL CHECK
# ============================================================

print("\n======================================")
print("FAST WD SETTINGS CHECK")
print("======================================")

for filename in [
    "inlist_common",
    "inlist_remove_env",
    "inlist",
]:
    path = RUN_DIR / filename

    if path.exists():
        print(f"\n--- {filename} ---")

        text = path.read_text()

        for parameter in [
            "initial_mass",
            "initial_z",
            "mesh_delta_coeff",
            "varcontrol_target",
            "max_years_for_timestep",
            "max_model_number",
            "log_Teff_upper_limit",
            "min_timestep_limit",
            "history_interval",
            "profile_interval",
            "terminal_interval",
        ]:
            matches = re.findall(
                rf"^\s*{re.escape(parameter)}\s*=.*$",
                text,
                flags=re.MULTILINE
            )

            for match in matches:
                print(match.strip())


print("\n======================================")
print("FAST WD INLISTS UPDATED")
print("======================================")
print("Run directory:", RUN_DIR)


UPDATING inlist_common
  inlist_common: initial_mass = 1.5d0
  inlist_common: initial_z = 0.0142d0
  inlist_common: mesh_delta_coeff = 1.0
  inlist_common: varcontrol_target = 0.01
  inlist_common: max_years_for_timestep = 1e+06
  inlist_common: history_interval = 100
  inlist_common: terminal_interval = 100

UPDATING inlist_remove_env
  inlist_remove_env: max_model_number = 30000
  inlist_remove_env: log_Teff_upper_limit = 3.8
  inlist_remove_env: min_timestep_limit = 1e-08
Replacements: 1
      ! limit max_model_number as part of test_suite
      max_model_number = 10000
Replacements: 1
Updated: /Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_FAST_1p5Msun_Z0p0142/inlist_settle
      ! limit max_model_number as part of test_suite
      max_model_number = 10000

FAST WD SETTINGS CHECK

--- inlist_common ---
initial_mass = 1.5d0 ! this default produces a ~0.6 Msun WD.
initial_z = 0.0142d0
mesh_delta_coeff = 1.0
varcontrol_target = 0.01
max_years_for_timestep = 1e+06
history_interv

In [40]:
from pathlib import Path
import re

# ============================================================
# WHITE DWARF RUN DIRECTORY
# ============================================================

RUN_DIR = Path(
    "/Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_FAST_1p5Msun_Z0p0142"
)

settle = RUN_DIR / "inlist_settle"

# ============================================================
# CHECK THAT INLIST_EXISTS
# ============================================================

if not settle.exists():
    raise FileNotFoundError(
        f"inlist_settle not found:\n{settle}"
    )

print("Found:")
print(settle)

# ============================================================
# READ INLIST
# ============================================================

text = settle.read_text()

# ============================================================
# CHANGE max_model_number
# ============================================================

text, n1 = re.subn(
    r"(^\s*max_model_number\s*=\s*)\d+",
    r"\g<1>10000",
    text,
    flags=re.MULTILINE
)

# ============================================================
# CHANGE log_L_lower_limit
# ============================================================

text, n2 = re.subn(
    r"(^\s*log_L_lower_limit\s*=\s*)[-+0-9.eEdD]+",
    r"\g<1>-3d0",
    text,
    flags=re.MULTILINE
)

# ============================================================
# CHECK THAT PARAMETERS WERE FOUND
# ============================================================

print("\nReplacements:")
print("max_model_number:", n1)
print("log_L_lower_limit:", n2)

if n1 == 0:
    print("\nWARNING: max_model_number was not found.")

if n2 == 0:
    print("WARNING: log_L_lower_limit was not found.")

# ============================================================
# WRITE CHANGES
# ============================================================

if n1 > 0 or n2 > 0:
    settle.write_text(text)
    print("\nUpdated inlist_settle.")
else:
    print("\nNo changes were made.")

# ============================================================
# VERIFY FINAL SETTINGS
# ============================================================

print("\n============================================================")
print("FINAL inlist_settle SETTINGS")
print("============================================================")

for line in settle.read_text().splitlines():

    stripped = line.strip()

    if not stripped:
        continue

    if stripped.startswith("!"):
        continue

    if (
        re.match(r"^\s*max_model_number\s*=", line)
        or re.match(r"^\s*log_L_lower_limit\s*=", line)
    ):
        print(line)

print("============================================================")
print("DONE")
print("============================================================")

Found:
/Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_FAST_1p5Msun_Z0p0142/inlist_settle

Replacements:
max_model_number: 1
log_L_lower_limit: 1

Updated inlist_settle.

FINAL inlist_settle SETTINGS
      max_model_number = 10000
      log_L_lower_limit = -3d0
DONE


In [41]:
# ============================================================
# CHECK MESA MODEL FILES
# ============================================================

model_files = [
    "zams.mod",
    "end_he_core_burn.mod",
    "co_core.mod",
    "co_wd.mod",
    "final.mod",
]

print("============================================================")
print("CHECKING MODEL FILES")
print("============================================================")

found = []

for filename in model_files:

    path = RUN_DIR / filename

    if path.exists():
        found.append(filename)
        print(f"  Found: {filename}")
    else:
        print(f"  Not found: {filename}")


print("\nModel files found:")
if found:
    for filename in found:
        print(" ", filename)
else:
    print("  None")


print("\n============================================================")
print("NO MODEL FILES WERE DELETED")
print("============================================================")

CHECKING MODEL FILES
  Not found: zams.mod
  Not found: end_he_core_burn.mod
  Found: co_core.mod
  Not found: co_wd.mod
  Not found: final.mod

Model files found:
  co_core.mod

NO MODEL FILES WERE DELETED


In [42]:
# ============================================================
# RUN FAST MESA WD
# ============================================================

import subprocess
from pathlib import Path
import time
import os

print("=" * 60)
print("STARTING FAST CO-WD MESA RUN")
print("=" * 60)

print("Run directory:", RUN_DIR)
print("Mass:", MASS_MSUN, "Msun")
print("Z:", INITIAL_Z)
print()

# ------------------------------------------------------------
# Check run directory
# ------------------------------------------------------------

RUN_DIR = Path(RUN_DIR)

if not RUN_DIR.exists():
    raise FileNotFoundError(
        f"Run directory does not exist:\n{RUN_DIR}"
    )

# ------------------------------------------------------------
# Check rn
# ------------------------------------------------------------

RN = RUN_DIR / "rn"

if not RN.exists():
    raise FileNotFoundError(
        f"Could not find rn:\n{RN}"
    )

RN.chmod(0o755)

print("✓ Run directory found")
print("✓ rn found")
print()

# ------------------------------------------------------------
# Make sure LOGS directory exists
# ------------------------------------------------------------

LOGS_DIR = RUN_DIR / "LOGS"
LOGS_DIR.mkdir(parents=True, exist_ok=True)

print("✓ LOGS directory ready")
print()

# ------------------------------------------------------------
# Remove old MESA log
# ------------------------------------------------------------

log_file = RUN_DIR / "mesa_run.log"

if log_file.exists():
    log_file.unlink()

# ------------------------------------------------------------
# Run MESA
# ------------------------------------------------------------

print("Running MESA...")
print()

start_time = time.time()

with open(log_file, "w") as log:

    result = subprocess.run(
        [str(RN)],
        cwd=str(RUN_DIR),
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
        env=os.environ.copy()
    )

elapsed = time.time() - start_time

# ------------------------------------------------------------
# Result
# ------------------------------------------------------------

print("=" * 60)
print("MESA FINISHED")
print("=" * 60)

print(f"Return code: {result.returncode}")
print(f"Runtime: {elapsed/60:.1f} minutes")
print(f"Full output: {log_file}")
print()

# ------------------------------------------------------------
# Check LOGS
# ------------------------------------------------------------

print("===== LOGS =====")

if LOGS_DIR.exists():

    print("✓ LOGS directory exists")

    history = LOGS_DIR / "history.data"

    if history.exists():
        print(
            f"✓ history.data "
            f"({history.stat().st_size / 1024:.1f} KB)"
        )
    else:
        print("✗ history.data not found")

    profiles = sorted(LOGS_DIR.glob("profile*.data"))

    print(f"Profiles found: {len(profiles)}")

    for p in profiles[-5:]:
        print(" ", p.name)

else:
    print("✗ LOGS directory does not exist")

print()

# ------------------------------------------------------------
# Check model files
# ------------------------------------------------------------

print("===== MODEL FILES =====")

models = [
    "zams.mod",
    "end_he_core_burn.mod",
    "co_core.mod",
    "co_wd.mod",
    "final.mod",
]

for model in models:

    path = RUN_DIR / model

    if path.exists():
        print(f"✓ {model}")
    else:
        print(f"✗ {model}")

print()

# ------------------------------------------------------------
# Show MESA output if run failed
# ------------------------------------------------------------

if result.returncode != 0:

    print("===== MESA OUTPUT =====")

    if log_file.exists():

        lines = log_file.read_text(
            errors="replace"
        ).splitlines()

        if lines:
            for line in lines[-100:]:
                print(line)
        else:
            print("mesa_run.log is empty.")

    else:
        print("mesa_run.log was not created.")

print()

print("=" * 60)
print("FAST CO-WD RUN COMPLETE")
print("=" * 60)

STARTING FAST CO-WD MESA RUN
Run directory: /Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_FAST_1p5Msun_Z0p0142
Mass: 1.5 Msun
Z: 0.0142

✓ Run directory found
✓ rn found

✓ LOGS directory ready

Running MESA...

MESA FINISHED
Return code: 1
Runtime: 435.6 minutes
Full output: /Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_FAST_1p5Msun_Z0p0142/mesa_run.log

===== LOGS =====
✓ LOGS directory exists
✓ history.data (336.4 KB)
Profiles found: 91
  profile88.data
  profile89.data
  profile9.data
  profile90.data
  profile91.data

===== MODEL FILES =====
✗ zams.mod
✗ end_he_core_burn.mod
✓ co_core.mod
✓ co_wd.mod
✗ final.mod

===== MESA OUTPUT =====
log_dt 1.793   age 4.299E+08   model 9946   iters    0   steps    5   retries    9   nzlo  789   nzhi 2131   n 1343   nz 2131   diffusion_call_number 1797
log_dt 1.782   age 4.299E+08   model 9947   iters    0   steps    5   retries    9   nzlo  786   nzhi 2128   n 1343   nz 2128   diffusion_call_number 1798
log_dt 1.778   age 4.299E+

In [7]:
star_path = RUN_DIR / "star"

print("===== STAR BEING USED =====")
print(star_path)

check = subprocess.run(
    ["otool", "-L", str(star_path)],
    capture_output=True,
    text=True
)

print(check.stdout)

assert "libhdf5_fortran.310.dylib" in check.stdout, \
    "STOP: Jupyter is using the wrong HDF5/STAR executable!"

===== STAR BEING USED =====
/Users/mchau/mesa_tutorial/WD/CO_WD_1p5Msun_Z0p0142/star
/Users/mchau/mesa_tutorial/WD/CO_WD_1p5Msun_Z0p0142/star:
	@rpath/lib/libcrmath.dylib (compatibility version 0.0.0, current version 0.0.0)
	@rpath/lib/libcrlibm.0.dylib (compatibility version 1.0.0, current version 1.0.0)
	@rpath/math-slots/crmath/lib/liblapack95.dylib (compatibility version 0.0.0, current version 0.0.0)
	@rpath/math-slots/crmath/lib/libtmg.dylib (compatibility version 0.0.0, current version 0.0.0)
	@rpath/math-slots/crmath/lib/liblapack.dylib (compatibility version 0.0.0, current version 0.0.0)
	@rpath/math-slots/crmath/lib/libblas.dylib (compatibility version 0.0.0, current version 0.0.0)
	@rpath/lib/libhdf5_fortran.310.dylib (compatibility version 314.0.0, current version 314.2.0)
	@rpath/lib/libhdf5.310.dylib (compatibility version 316.0.0, current version 316.1.0)
	/usr/lib/libz.1.dylib (compatibility version 1.0.0, current version 1.2.11)
	@rpath/lib/libpgplot.dylib (compatibilit

In [17]:
from pathlib import Path

WD_ROOT = Path("/Users/mchau/mesa_tutorial/WHITE_DWARF/WD")

for filename in ["zams.mod", "end_he_core_burn.mod", "co_core.mod", "co_wd.mod", "final.mod"]:
    print(f"\n===== {filename} =====")
    matches = list(WD_ROOT.rglob(filename))
    if matches:
        for p in matches:
            print(p)
    else:
        print("NOT FOUND")


===== zams.mod =====
/Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_REGENERATE_1p5Msun_Z0p0142/zams.mod
/Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_1p5Msun_Z0p0142/zams.mod

===== end_he_core_burn.mod =====
/Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_1p5Msun_Z0p0142/end_he_core_burn.mod

===== co_core.mod =====
/Users/mchau/mesa_tutorial/WHITE_DWARF/WD/make_co_wd_fast/co_core.mod
/Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_1p5Msun_Z0p0142/co_core.mod
/Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_FAST_1p5Msun_Z0p0142/co_core.mod

===== co_wd.mod =====
NOT FOUND

===== final.mod =====
NOT FOUND


In [15]:
for filename in ["co_wd.mod", "final.mod"]:
    path = RUN_DIR / filename
    if path.exists():
        path.unlink()
        print(f"Removed {filename}")